# Event Labeling

Calculate triple-barrier direction labels from the integrated candidate schema. Horizontal barriers use the exponentially weighted standard deviation of returns spanning 50 dollar bars with a span of 100 bar observations, aligned with the 50-bar vertical barrier. Label thresholds and retained classes are derived from development only, while missing feature values remain unchanged for the later `Clean the Data` stage.


## Process the Data


In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve().parents[1]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_preprocessing.event_labeling import build_labeled_event_data

period = "2025-01-01_2025-12-31"
event_dir = PROJECT_ROOT / "data/research_data/events"
candidate_path = event_dir / f"aapl_news_candidate_split_{period}.parquet"
dollar_path = PROJECT_ROOT / f"data/research_data/market/features/aapl_dollar_bar_{period}.parquet"
event_path = event_dir / f"aapl_news_primary_model_{period}.parquet"
partition_path = event_dir / f"aapl_news_labeled_split_{period}.parquet"

candidate_split = pd.read_parquet(candidate_path).sort_values("event_start", ignore_index=True)
dollar_bars = pd.read_parquet(dollar_path).sort_values("end").drop_duplicates("end", keep="last")


In [2]:
model_data, partition_manifest = build_labeled_event_data(candidate_split, dollar_bars)

assert model_data.shape[1] == 60
assert model_data["event_start"].is_unique
assert set(model_data["direction_label"]) == {-1, 1}

event_dir.mkdir(parents=True, exist_ok=True)
model_data.to_parquet(event_path, index=False)
partition_manifest.to_parquet(partition_path, index=False)
print(event_path)
print(partition_path)


2026-08-19 23:09:57.226 | DEBUG    | src.data_preprocessing.event_labeling:drop_labels:260 - Dropping label 0.0 with frequency 0.0010204081632653062.


/Users/kwonjunhyuk9/Documents/financial-machine-learning/data/research_data/events/aapl_news_primary_model_2025-01-01_2025-12-31.parquet
/Users/kwonjunhyuk9/Documents/financial-machine-learning/data/research_data/events/aapl_news_labeled_split_2025-01-01_2025-12-31.parquet


## Take a Quick Look at the Data Structure


In [3]:
development_starts = partition_manifest.loc[partition_manifest["partition"].eq("development"), "event_start"]
development_data = model_data[model_data["event_start"].isin(development_starts)]
development_data.head()


,event_start,symbol,event_end,vertical_barrier,target_return,raw_return,direction_label,mean_sentiment_score,fractionally_differenced_log_close,McClellan Oscillator,...,Bollinger Band Middle,Bollinger Band Lower,True Range,Average True Range,Keltner Channel Upper,Keltner Channel Middle,Keltner Channel Lower,Donchian Channel Upper,Donchian Channel Middle,Donchian Channel Lower
0,2025-01-02 15:00:32.232433+00:00,AAPL,2025-01-02 15:24:06.857548+00:00,2025-01-02 15:24:06.857548+00:00,0.007515,-0.005267,-1,0.090007,1.455318,2.6921,...,246.6546,246.1924,0.28,0.1921,247.0100,246.6257,246.2414,246.99,246.570,246.15
1,2025-01-02 15:32:28.839475+00:00,AAPL,2025-01-02 15:49:31.568019+00:00,2025-01-02 15:49:31.568019+00:00,0.006929,-0.000061,-1,0.914739,1.453821,5.2764,...,245.4696,244.9892,0.32,0.1929,245.8153,245.4296,245.0439,245.88,245.420,244.96
2,2025-01-02 16:48:23.973430+00:00,AAPL,2025-01-02 17:24:27.805471+00:00,2025-01-02 17:27:48.251810+00:00,0.002947,-0.003145,-1,-0.024872,1.452908,0.9699,...,244.8911,244.7601,0.09,0.1161,245.1103,244.8782,244.6460,245.15,244.940,244.73
3,2025-01-02 17:35:50.819642+00:00,AAPL,2025-01-02 18:07:35.869987+00:00,2025-01-02 18:07:35.869987+00:00,0.003470,-0.000124,-1,0.052554,1.450704,-3.5400,...,243.1082,242.7012,0.17,0.1657,243.4921,243.1607,242.8292,243.49,243.105,242.72
4,2025-01-03 15:00:13.717905+00:00,AAPL,2025-01-03 15:09:26.860772+00:00,2025-01-03 15:31:35.970761+00:00,0.003829,0.003910,1,0.851262,1.451799,0.0410,...,242.8468,242.4094,0.44,0.2918,243.4799,242.8963,242.3127,243.15,242.765,242.38


In [4]:
development_data.info()


<class 'pandas.DataFrame'>
RangeIndex: 978 entries, 0 to 977
Data columns (total 60 columns):
 #   Column                                    Non-Null Count  Dtype              
---  ------                                    --------------  -----              
 0   event_start                               978 non-null    datetime64[us, UTC]
 1   symbol                                    978 non-null    str                
 2   event_end                                 978 non-null    datetime64[us, UTC]
 3   vertical_barrier                          978 non-null    datetime64[us, UTC]
 4   target_return                             978 non-null    float64            
 5   raw_return                                978 non-null    float64            
 6   direction_label                           978 non-null    int8               
 7   mean_sentiment_score                      978 non-null    float64            
 8   fractionally_differenced_log_close        978 non-null    float64      

In [5]:
development_data["direction_label"].value_counts()


direction_label
 1    497
-1    481
Name: count, dtype: int64

In [6]:
development_data.select_dtypes(include="number").describe()


,target_return,raw_return,direction_label,mean_sentiment_score,fractionally_differenced_log_close,McClellan Oscillator,Advancers - Decliners,On-Balance Volume,Accumulation/Distribution Line,Chaikin Oscillator,...,Bollinger Band Middle,Bollinger Band Lower,True Range,Average True Range,Keltner Channel Upper,Keltner Channel Middle,Keltner Channel Lower,Donchian Channel Upper,Donchian Channel Middle,Donchian Channel Lower
count,978.000000,978.000000,978.000000,978.000000,978.000000,978.000000,978.000000,9.780000e+02,9.780000e+02,978.000000,...,978.000000,978.000000,978.000000,978.000000,978.000000,978.000000,978.000000,978.000000,978.000000,978.000000
mean,0.004169,-0.000043,0.016360,0.064051,1.420276,0.342319,213.048599,9.929208e+05,1.959475e+06,39.640262,...,217.351548,216.773302,0.544836,0.202234,217.752856,217.348387,216.943920,217.907188,217.322628,216.738067
std,0.002471,0.004409,1.000378,0.510390,0.022969,3.635202,35.068508,3.787733e+05,8.172938e+05,1802.555068,...,18.298044,18.403244,1.308237,0.144394,18.267543,18.296588,18.330139,18.248506,18.312534,18.414184
min,0.002162,-0.042585,-1.000000,-0.968094,1.334276,-30.654700,0.000000,-1.691100e+04,-1.931217e+04,-6072.855700,...,172.579600,171.504900,0.030000,0.046100,173.507800,172.470000,171.206400,173.405000,171.012500,168.620000
25%,0.002795,-0.003142,-1.000000,-0.048831,1.403124,-0.022025,202.211250,7.354570e+05,1.574784e+06,-1172.603600,...,202.557500,202.155300,0.130000,0.136100,202.913825,202.613550,202.247500,203.286250,202.808750,202.212500
50%,0.003547,0.000126,1.000000,0.052554,1.415579,1.083950,212.085000,1.017142e+06,1.957313e+06,59.930550,...,212.376800,211.899250,0.195000,0.167500,212.750100,212.353150,212.059750,212.830000,212.310000,211.840000
75%,0.004666,0.003245,1.000000,0.318801,1.438827,2.247000,231.573750,1.348392e+06,2.694213e+06,1261.081725,...,231.951150,231.245550,0.295000,0.217900,232.288675,231.982950,231.574100,232.295000,231.973750,231.240000
max,0.040082,0.014305,1.000000,0.941135,1.469257,11.868500,257.870000,1.627238e+06,3.137575e+06,6516.393900,...,258.257900,257.617500,13.990000,1.958200,258.572600,258.089700,257.606800,258.780000,258.230000,257.680000


In [7]:
development_data.select_dtypes(include="number").replace([np.inf, -np.inf], np.nan).hist(figsize=(20, 24), bins=30)


array([[<Axes: title={'center': 'target_return'}>,
        <Axes: title={'center': 'raw_return'}>,
        <Axes: title={'center': 'direction_label'}>,
        <Axes: title={'center': 'mean_sentiment_score'}>,
        <Axes: title={'center': 'fractionally_differenced_log_close'}>,
        <Axes: title={'center': 'McClellan Oscillator'}>,
        <Axes: title={'center': 'Advancers - Decliners'}>],
       [<Axes: title={'center': 'On-Balance Volume'}>,
        <Axes: title={'center': 'Accumulation/Distribution Line'}>,
        <Axes: title={'center': 'Chaikin Oscillator'}>,
        <Axes: title={'center': 'New Highs - New Lows'}>,
        <Axes: title={'center': 'Money Flow Index'}>,
        <Axes: title={'center': 'Williams %R'}>,
        <Axes: title={'center': 'Aroon Indicator Up'}>],
       [<Axes: title={'center': 'Aroon Indicator Down'}>,
        <Axes: title={'center': 'Commodity Channel Index'}>,
        <Axes: title={'center': 'Relative Vigor Index'}>,
        <Axes: title={'cen